<a href="https://colab.research.google.com/github/Eliezer-Carvalho/Adamastor-GPT/blob/main/Adamastor_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Adamastor-GPT**

Este modelo é baseado na arquitetura <b> Transformer </b>, inspirado nos modelos do tipo <b> GPT (<i>Generative Pre-trained Transformer</i>)</b>. <br>

Ao longo deste Colab, iremos navegar pelas componentes cruciais dos modelos GPT-like, que atualmente dominam grande parte do mercado em Inteligência Artificial.

<b> No final, teremos um modelo estilo GPT funcional, capaz de realizar inferência de forma limpa e consistente. </b>

In [11]:
from tokenizers import Tokenizer
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu" #Mover tudo para a GPU

In [12]:
#Dados do Modelo

Sequence_Length = 32
Batch_Size = 16
Embedding_Dimension = 64
Head_Size = 8 #Head_Size = Embedding_Dimension // Number_Head
Number_Head = 8

##### Tokenization



*   Aplicação do Tokenizer
*   Colocação dos Ids num Tensor
*   Separação dados de treino e dados de teste



In [13]:
Tokenizer = Tokenizer.from_file ("Tokenizer - GAMA")

with open ("Os Lusiadas.txt", "r", encoding = "utf-8") as f:
  dataset = f.read()

Tokenization = Tokenizer.encode (dataset)
print (len(Tokenization.ids))
print (len(Tokenization.tokens))

tensor = torch.tensor (Tokenization.ids, dtype = torch.long)
print (tensor [:50])

n = int (0.9 * (len(tensor)))
dados_treino = tensor [:n] #90% Dados de Treino
dados_teste = tensor [n:] #10% Dados de Teste

214909
214909
tensor([  40,   44,    5,   37,   46,   44,   83,   27,   30,   27,   44,  108,
          37, 2174,   69,    5,   55,   56,    5,   29,  246,  101,  161,  108,
          29,  298,   65,    5,   35,  108,   27,   69,    5,  219,   63,  177,
           5,   56,    5,  142,    5,   28,  219,  101,  161,    5,  541,  233,
         261,  227])


##### Batch e Sequence Length

In [14]:
torch.manual_seed (2000)

def Batch ():
  idx = torch.randint (0, len(dados_treino) - Sequence_Length, (Batch_Size, ))
  x = torch.stack ([dados_treino [i: i + Sequence_Length] for i in idx])
  y = torch.stack ([dados_treino [i + 1: i + Sequence_Length + 1] for i in idx])

  return x, y

x, y = Batch()

x = x.to(device)
print (x.shape) #(B, T)
print (x[1])

y = y.to(device)
print (y.shape) #(B, T)
print (y[1])

torch.Size([16, 32])
tensor([  66,  255,   65,    5,   63,  867,   65,    5,   69,   56,    5,  261,
        1394, 4747,  106,   29,  236,    5,   57,  246,   52,    5,   69,   71,
         177,    5,  420,  160,   69,    5,   72,  261], device='cuda:0')
torch.Size([16, 32])
tensor([ 255,   65,    5,   63,  867,   65,    5,   69,   56,    5,  261, 1394,
        4747,  106,   29,  236,    5,   57,  246,   52,    5,   69,   71,  177,
           5,  420,  160,   69,    5,   72,  261, 1487], device='cuda:0')


###


---



In [15]:
class Head (nn.Module): #Head = Attention
  def __init__(self):
    super().__init__()
    self.Query = nn.Linear (Embedding_Dimension, Head_Size, bias = False)
    self.Key = nn.Linear (Embedding_Dimension, Head_Size, bias = False)
    self.Value = nn.Linear (Embedding_Dimension, Head_Size, bias = False)

  def forward (self, x):
    B, T, C = x.shape

    Q = self.Query (x)
    K = self.Key (x)
    V = self.Value (x)

    attention_score = Q @ K.transpose (-2, -1) * (Head_Size ** -0.5)

    tril = torch.tril (torch.ones (T, T, device = x.device))
    attention_score = attention_score.masked_fill (tril == 0, float ("-inf"))

    attention_score = F.softmax (attention_score, dim = -1)

    output = attention_score @ V
    return output


class Multi_Head (nn.Module):
  def __init__(self):
    super().__init__()
    self.Concat = nn.ModuleList ([Head() for i in range (Number_Head)]) #Head Concatenation #Necessário para Multi Head Attention
    self.Output_Projection = nn.Linear (Embedding_Dimension, Embedding_Dimension) #Necessário para misturar a informação das Heads #Desta forma volta também à dimensão (B, T, C)

  def forward (self, x):

    output = self.Output_Projection (torch.cat ([h (x) for h in self.Concat], dim = -1))
    return output


In [16]:
class Block (nn.Module):
  def __init__(self):
    super().__init__()

    self.LayerNorm_Attention = nn.LayerNorm (Embedding_Dimension)
    self.Masked_Multi_Head_Attention = Multi_Head()


    self.FeedForward_NeuralNet = nn.Sequential (
        nn.Linear (Embedding_Dimension, 4 * Embedding_Dimension),
        nn.ReLU(),
        nn.Linear (Embedding_Dimension * 4, Embedding_Dimension)
    )
    self.LayerNorm_FNN = nn.LayerNorm (Embedding_Dimension)

  def forward (self, x):
    Norm_Attention = self.Masked_Multi_Head_Attention (self.LayerNorm_Attention (x)) #Norm #LayerNorm
    Add_Attention = x + Norm_Attention #Add #Residual Connection
    x = Add_Attention

    Norm_FNN = self.FeedForward_NeuralNet (self.LayerNorm_FNN (x)) #Norm #LayerNorm
    Add_FNN = x + Norm_FNN #Add #Residual Connection
    x = Add_FNN

    return x


In [17]:
class ADAMASTOR_GPT (nn.Module):
  def __init__(self):
    super().__init__()
    self.Embedding = nn.Embedding (Tokenizer.get_vocab_size(), Embedding_Dimension)
    self.Positional_Encoding = nn.Embedding (Sequence_Length, Embedding_Dimension)
    self.Dropout = nn.Dropout (0.1) #Diferença em relação ao Transformer original - Adiciona-se Dropout para evitar overfitting - Desliga alguns neurónios aleatóriamente

    self.Blocks = nn.Sequential (
        Block(),
        Block(),
        Block(),
        Block()
    )

    self.Language_Modeling = nn.Linear (Embedding_Dimension, Tokenizer.get_vocab_size())

  def forward (self, x):
    B, T = x.shape

    x = self.Dropout(self.Embedding (x) + self.Positional_Encoding (torch.arange (T, device = x.device)))

    x = self.Blocks(x)

    logits = self.Language_Modeling (x)

    return logits #Não é necessário aplicar Softmax no treino porque Cross Entropy já o faz



In [18]:
if __name__ == "__main__":

  modelo = ADAMASTOR_GPT().to(device)
  optimizer = torch.optim.AdamW (modelo.parameters(), lr = 3e-4)
  loss_function = nn.CrossEntropyLoss()

  for i in range (50000):

    x, y = Batch()
    x, y = x.to(device), y.to(device)

    logits = modelo (x)

    B, T, V = logits.shape

    loss = loss_function(
        logits.view (B * T, V),
        y.view (B * T)
    )

    optimizer.zero_grad()
    loss.backward()

    optimizer.step()

    if i % 100 == 0:
        print(i, loss.item())



0 8.936691284179688
100 4.066038131713867
200 3.7298643589019775
300 3.512319803237915
400 3.267977237701416
500 3.288712978363037
600 3.1048197746276855
700 3.135470151901245
800 3.0391829013824463
900 3.1331536769866943
1000 3.0897583961486816
1100 3.1657662391662598
1200 3.0381569862365723
1300 3.134751558303833
1400 2.870006561279297
1500 3.154853343963623
1600 2.9916367530822754
1700 3.0312676429748535
1800 2.742114543914795
1900 3.072033643722534
2000 2.892232894897461
2100 2.7815942764282227
2200 2.779970645904541
2300 2.9049277305603027
2400 2.9157447814941406
2500 2.910132884979248
2600 2.7041244506835938
2700 2.704080820083618
2800 2.7870781421661377
2900 2.762263774871826
3000 2.768110513687134
3100 2.8293418884277344
3200 2.744985580444336
3300 2.6669061183929443
3400 2.602895736694336
3500 2.843937873840332
3600 2.8054895401000977
3700 2.7771241664886475
3800 2.6469664573669434
3900 2.6962199211120605
4000 2.801409959793091
4100 2.6411232948303223
4200 2.7252345085144043
4

In [23]:
def generate(model, idx, max_new_tokens, temperature=1.0):
    modelo.eval()
    device = next(model.parameters()).device  # pega device do modelo
    idx = idx.to(device)

    for _ in range(max_new_tokens):

        idx_cond = idx[:, -Sequence_Length:]

        logits = model(idx_cond)
        logits = logits[:, -1, :] / temperature

        probs = F.softmax(logits, dim=-1)

        next_token = torch.multinomial(probs, num_samples=1)

        idx = torch.cat([idx, next_token], dim=1)

    return idx

start = torch.zeros((1, 1), dtype=torch.long).to(device)
output = generate(modelo, start, 100)

text = Tokenizer.decode(output[0].tolist())
print(text)


#Problema no Tokenizer
#Continuar a testar, escrever explicação do códigop

o   s ec ret o .   N ã o   m as   s e   s ust ent av a ,  e   d o   R ei   e   p u rav a   M art e   I l iss e   H erc ic l in a ? »   « A ss i   d e   J eit o   T emp o   e   al i   v ej ar - s e   s e   ê m   ent end er rib a ,  O   qu
